In [18]:
#=================================================================
# Celda 1 — Setup + inspeccionar JSONs disponibles
#=================================================================
import pandas as pd
import numpy as np
from pathlib import Path
import os
os.chdir("/workspaces/TFG_Spain-s_Migratory_Flow")

# ── Rutas siguiendo el patrón del proyecto ───────────────────
RAW_DIR    = Path("output/economico/01-raw")
SILVER_DIR = Path("output/economico/02-silver")
SILVER_DIR.mkdir(parents=True, exist_ok=True)

# Cargar parquets generados por 01_economico
parquets = sorted(RAW_DIR.glob("*.parquet"))
assert len(parquets) > 0, f"❌ No hay parquets en {RAW_DIR}"

print(f"📂 Parquets encontrados:")
for p in parquets:
    print(f"   {p.name}")

# Cargar todos en un dict por nombre de tabla
dfs_raw = {}
for p in parquets:
    nombre = p.stem  # nombre sin extensión = nombre de tabla
    dfs_raw[nombre] = pd.read_parquet(p)
    print(f"✅ {nombre}: {dfs_raw[nombre].shape}")

📂 Parquets encontrados:
   raw_centros_educativos_ccaa.parquet
   raw_compraventas_vivienda.parquet
   raw_densidad_poblacion.parquet
   raw_hipotecas_vivienda.parquet
   raw_ipv_precio_vivienda_anual.parquet
   raw_ipv_precio_vivienda_ccaa.parquet
   raw_renta_media_hogar.parquet
   raw_tasa_paro_ccaa.parquet
   raw_turismo_pernoctaciones.parquet
   raw_turismo_viajeros.parquet
✅ raw_centros_educativos_ccaa: (180, 9)
✅ raw_compraventas_vivienda: (82440, 14)
✅ raw_densidad_poblacion: (3975, 12)
✅ raw_hipotecas_vivienda: (4460, 12)
✅ raw_ipv_precio_vivienda_anual: (2128, 11)
✅ raw_ipv_precio_vivienda_ccaa: (17024, 11)
✅ raw_renta_media_hogar: (576, 12)
✅ raw_tasa_paro_ccaa: (5040, 10)
✅ raw_turismo_pernoctaciones: (131670, 13)
✅ raw_turismo_viajeros: (136910, 14)


In [19]:
#=================================================================
# Celda 2 — Parser genérico de tablas INE
#=================================================================
def parse_serie_ine(serie: dict) -> list:
    """
    Convierte una serie INE {COD, Nombre, MetaData, Data} en lista de filas planas.
    MetaData → variables descriptivas (provincia, tipo, etc.)
    Data     → valores temporales {Fecha, Valor, Anyo}
    """
    meta = {}
    for m in serie.get("MetaData", []):
        var = m.get("T3_Variable", m.get("Variable", {}).get("Nombre", "var"))
        val = m.get("Nombre", "")
        meta[var] = val

    rows = []
    for d in serie.get("Data", []):
        row = {**meta}
        row["valor"]   = d.get("Valor")
        row["fecha"]   = d.get("Fecha", "")
        row["anyo"]    = d.get("Anyo",  str(row["fecha"])[:4] if row["fecha"] else None)
        row["secreto"] = d.get("Secreto", False)
        rows.append(row)
    return rows

def parse_tabla(registros: list) -> pd.DataFrame:
    rows = []
    for serie in registros:
        rows.extend(parse_serie_ine(serie))
    return pd.DataFrame(rows)

# Test rápido
for nombre, registros in tablas_ok.items():
    df_tmp = parse_tabla(registros)
    print(f"  {nombre}: {df_tmp.shape} | cols: {list(df_tmp.columns)}")

  tasa_paro_actividad_provincia: (45792, 9) | cols: ['Tipo de dato', 'Sexo', 'Total Nacional', 'Nacionalidad', 'valor', 'fecha', 'anyo', 'secreto', 'Provincias']
  ipv_precio_vivienda_ccaa: (17024, 8) | cols: ['Total Nacional', 'General, vivienda nueva y de segunda mano', 'Índices y Tasas', 'valor', 'fecha', 'anyo', 'secreto', 'Comunidades y Ciudades Autónomas']
  compraventas_vivienda: (82440, 11) | cols: ['Total Nacional', 'Estado de la vivienda', 'Título de adquisición', 'Tipo de dato', 'valor', 'fecha', 'anyo', 'secreto', 'Régimen de la vivienda', 'Comunidades y Ciudades Autónomas', 'Provincias']
  hipotecas_vivienda: (4460, 9) | cols: ['Naturaleza de la finca', 'Concepto financiero', 'Total Nacional', 'Tipo de muestreo', 'valor', 'fecha', 'anyo', 'secreto', 'Comunidades y Ciudades Autónomas']
  renta_media_hogar: (576, 9) | cols: ['Totales Territoriales', 'Tipo de hogar', 'SALDOS CONTABLES', 'Totales de edad', 'Cambios de conceptos', 'valor', 'fecha', 'anyo', 'secreto']
  renta_me

In [20]:
#=================================================================
# Celda 3 — Limpiar tasa de paro por provincia × año
#=================================================================
df_paro_raw = parse_tabla(tablas_ok["tasa_paro_actividad_provincia"])
print(df_paro_raw.head(3).to_string())
print(f"\nColumnas: {list(df_paro_raw.columns)}")
print(f"Tipos de dato únicos: {df_paro_raw.get('Tipo de dato', pd.Series()).unique()[:5]}")
print(f"Sexo únicos: {df_paro_raw.get('Sexo', pd.Series()).unique()}")

        Tipo de dato         Sexo  Total Nacional Nacionalidad  valor                          fecha  anyo  secreto Provincias
0  Tasa de actividad  Ambos sexos  Total Nacional        Total  58.94  2025-10-01T00:00:00.000+02:00  2025    False        NaN
1  Tasa de actividad  Ambos sexos  Total Nacional        Total  59.30  2025-07-01T00:00:00.000+02:00  2025    False        NaN
2  Tasa de actividad  Ambos sexos  Total Nacional        Total  59.03  2025-04-01T00:00:00.000+02:00  2025    False        NaN

Columnas: ['Tipo de dato', 'Sexo', 'Total Nacional', 'Nacionalidad', 'valor', 'fecha', 'anyo', 'secreto', 'Provincias']
Tipos de dato únicos: <ArrowStringArray>
[             'Tasa de actividad',   'Tasa de paro de la población',
 'Tasa de empleo de la población']
Length: 3, dtype: str
Sexo únicos: <ArrowStringArray>
['Ambos sexos', 'Hombres', 'Mujeres']
Length: 3, dtype: str


In [21]:
#=================================================================
# Celda 4 — Limpiar tasa de paro
#=================================================================
df_paro = df_paro_raw.copy()

# Detectar columna de provincia automáticamente
prov_col = next((c for c in df_paro.columns 
                 if any(x in c.lower() for x in ["provincia", "provincias"])), None)
sexo_col = next((c for c in df_paro.columns 
                 if "sexo" in c.lower()), None)
tipo_col = next((c for c in df_paro.columns 
                 if "tipo" in c.lower()), None)

print(f"Columna provincia: {prov_col}")
print(f"Columna sexo:      {sexo_col}")
print(f"Columna tipo dato: {tipo_col}")

# Filtros
mask = pd.Series([True] * len(df_paro), index=df_paro.index)
if sexo_col:
    mask &= df_paro[sexo_col].str.contains("Ambos", na=False)
if tipo_col:
    mask &= df_paro[tipo_col].str.contains("paro", case=False, na=False)
mask &= df_paro["secreto"].eq(False)
mask &= df_paro["valor"].notna()

df_paro_clean = (df_paro[mask]
    .rename(columns={prov_col: "Provincia"})
    .assign(año=lambda x: pd.to_numeric(x["anyo"], errors="coerce").astype("Int64"))
    .query("Provincia != 'Total Nacional' and Provincia == Provincia")
    .groupby(["Provincia", "año"], as_index=False)["valor"]
    .mean()
    .rename(columns={"valor": "tasa_paro"})
    .dropna(subset=["Provincia", "año"])
)

print(f"\n✅ tasa_paro: {df_paro_clean.shape}")
print(f"   Años: {sorted(df_paro_clean['año'].dropna().unique())}")
print(f"   Provincias: {df_paro_clean['Provincia'].nunique()}")
print(df_paro_clean.head())

Columna provincia: Provincias
Columna sexo:      Sexo
Columna tipo dato: Tipo de dato

✅ tasa_paro: (1248, 3)
   Años: [np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
   Provincias: 52
  Provincia   año  tasa_paro
0  Albacete  2002      6.935
1  Albacete  2003      7.700
2  Albacete  2004      9.005
3  Albacete  2005      9.840
4  Albacete  2006     10.015


In [22]:
#=================================================================
# Celda 5 — Renta media por provincia × año
#=================================================================
df_renta_raw = parse_tabla(tablas_ok["renta_media_hogar_provincia"])

prov_col_r = next((c for c in df_renta_raw.columns 
                   if any(x in c.lower() for x in ["provincia", "territorial", "municipio"])), None)
print(f"Columna provincia renta: {prov_col_r}")
print(df_renta_raw.head(3).to_string())

# Filtrar solo nivel provincia (excluir municipios y nacional)
# La columna suele ser "Totales Territoriales" con valores tipo "Valencia/València"
df_renta_clean = (df_renta_raw
    .rename(columns={prov_col_r: "Provincia"})
    .assign(año=lambda x: pd.to_numeric(x["anyo"], errors="coerce").astype("Int64"))
    .query("Provincia != 'Total Nacional' and Provincia == Provincia")
    [["Provincia", "año", "valor"]]
    .dropna()
    .groupby(["Provincia", "año"], as_index=False)["valor"]
    .mean()
    .rename(columns={"valor": "renta_media"})
)

print(f"\n✅ renta_media: {df_renta_clean.shape}")
print(f"   Años: {sorted(df_renta_clean['año'].dropna().unique())}")
print(df_renta_clean.head())

Columna provincia renta: Totales Territoriales
  Totales Territoriales Semiintervalos de edad TOTALES CLASIFICACION EDUCACIÓN Porcentajes de renta por unidad de consumo Tipo de dato Cambios de conceptos  valor                          fecha  anyo  secreto AGRUPACIONES NIVELES DE FORMACIÓN
0        Total Nacional          16 y más años                           Total                               Primer decil     Personas            Base 2013    8.9  2025-01-01T00:00:00.000+01:00  2025    False                               NaN
1        Total Nacional          16 y más años                           Total                               Primer decil     Personas            Base 2013    8.9  2024-01-01T00:00:00.000+01:00  2024    False                               NaN
2        Total Nacional          16 y más años                           Total                               Primer decil     Personas            Base 2013    8.9  2023-01-01T00:00:00.000+01:00  2023    False                

In [23]:
#=================================================================
# Celda 6 — IPV precio vivienda (CCAA → complementario)
#=================================================================
df_ipv_raw = parse_tabla(tablas_ok["ipv_precio_vivienda_anual"])

# IPV es índice anual nacional/CCAA, no provincia → lo usamos como variable extra
df_ipv_clean = (df_ipv_raw
    .assign(año=lambda x: pd.to_numeric(x["anyo"], errors="coerce").astype("Int64"))
    [["año", "valor"]]
    .dropna()
    .groupby("año", as_index=False)["valor"]
    .mean()
    .rename(columns={"valor": "ipv_vivienda"})
)

print(f"✅ ipv_vivienda (nacional): {df_ipv_clean.shape}")
print(df_ipv_clean.head())

✅ ipv_vivienda (nacional): (19, 2)
    año  ipv_vivienda
0  2007     80.635545
1  2008     74.975741
2  2009     67.926589
3  2010     67.936268
4  2011     60.288830


In [24]:
#=================================================================
# Celda 7 — Merge final provincia × año + normalización
#=================================================================
from functools import reduce

# Merge tasa_paro + renta (ambas por provincia×año)
df_eco = pd.merge(df_paro_clean, df_renta_clean, on=["Provincia", "año"], how="outer")

# IPV es nacional → join por año
df_eco = pd.merge(df_eco, df_ipv_clean, on="año", how="left")

# Normalizar nombre provincia
df_eco["Provincia"] = df_eco["Provincia"].str.strip().str.title()
df_eco["año"] = df_eco["año"].astype("Int64")

# Ordenar
df_eco = df_eco.sort_values(["Provincia", "año"]).reset_index(drop=True)

print(f"📊 Panel económico final: {df_eco.shape}")
print(f"   Años:       {sorted(df_eco['año'].dropna().unique())}")
print(f"   Provincias: {df_eco['Provincia'].nunique()}")
print(f"\n   Nulos por columna:\n{df_eco.isnull().sum()}")
print(f"\n{df_eco.head(12).to_string()}")

📊 Panel económico final: (1248, 5)
   Años:       [np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
   Provincias: 52

   Nulos por columna:
Provincia          0
año                0
tasa_paro          0
renta_media     1248
ipv_vivienda     260
dtype: int64

   Provincia   año  tasa_paro  renta_media  ipv_vivienda
0   Albacete  2002     6.9350          NaN           NaN
1   Albacete  2003     7.7000          NaN           NaN
2   Albacete  2004     9.0050          NaN           NaN
3   Albacete  2005     9.8400          NaN           NaN
4   Albacete  2006    10.0150          NaN           NaN
5   Albacete  2007     9.1625          NaN     80.635545
6   Al

In [25]:
#=================================================================
# Celda 8 — Export (Guardado)
#=================================================================
ruta = SILVER_DIR / "economico_provincia_año.parquet"
df_eco.to_parquet(ruta, index=False)

# También CSV por compatibilidad con merge maestro
df_eco.to_csv(SILVER_DIR / "economico_provincia_año.csv", index=False)

print(f"✅ Guardado en: {SILVER_DIR}")
print(f"   Shape final: {df_eco.shape}")



✅ Guardado en: output/economico/02-silver
   Shape final: (1248, 5)
